# 3.05 Qdrant Vector Store

**Qdrant**는 Rust로 작성된 벡터 검색 엔진. `:memory:` / 파일 / Docker / Qdrant Cloud 네 가지 배포 모드가 있고, 같은 `QdrantClient` 클래스로 전환된다.

## 학습 목표

- `QdrantClient(':memory:')` 로 외부 서비스 없이 실습
- `QdrantVectorStore.from_documents` · `.from_texts`
- `qdrant_client.models.Filter` DSL로 복합 조건 필터
- 하이브리드 검색 (`RetrievalMode.HYBRID` + `FastEmbedSparse`)

## 언제 쓰나

- 셀프 호스팅 + 관리형 선택권이 필요할 때
- **sparse + dense 하이브리드** 검색을 벡터 스토어 레벨에서 해결하고 싶을 때
- gRPC 성능·대규모 payload 쿼리

## 3.05.1 환경 설정 / 서비스 연결

in-memory 모드는 추가 서비스 설치가 필요 없다. probe 셀은 `QdrantClient(':memory:').get_collections()` 으로 health check.

In [ ]:
# !pip install -U langchain-qdrant 'qdrant-client[fastembed]' langchain-openai

import os
from dotenv import load_dotenv
load_dotenv(override=True)

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 필요 (임베딩용)"

from qdrant_client import QdrantClient
# 실제 연결 테스트 (:memory: 도 내부적으로 클라이언트 초기화를 수행한다)
QdrantClient(":memory:").get_collections()
print("Qdrant in-memory OK")

## 3.05.2 기본 사용법

가장 간단한 경로: `QdrantVectorStore.from_documents(docs, embedding, location=':memory:', collection_name=...)`. 내부적으로 클라이언트 생성·컬렉션 생성·업서트를 한 번에 처리한다.

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

docs = [
    Document(page_content="Qdrant는 Rust 기반 벡터 검색 엔진이다.", metadata={"source": "qdrant", "lang": "ko"}),
    Document(page_content="RAG는 검색 증강 생성으로 LLM 응답 품질을 높인다.", metadata={"source": "blog", "lang": "ko"}),
    Document(page_content="FastEmbed는 sparse embeddings 지원 라이브러리이다.", metadata={"source": "fastembed", "lang": "en"}),
    Document(page_content="HNSW 인덱스는 Qdrant의 기본 ANN 구조이다.", metadata={"source": "docs", "lang": "ko"}),
]

store = QdrantVectorStore.from_documents(
    docs,
    embedding=embeddings,
    location=":memory:",
    collection_name="vs_demo",
)
for d in store.similarity_search("Rust 벡터 엔진", k=3):
    print("-", d.metadata["source"], "|", d.page_content[:40])

## 3.05.3 메타데이터 필터링 — `models.Filter`

Qdrant 필터는 dict가 아니라 **타입 안정적인 객체 모델**이다. 복잡한 조건도 `must` / `should` / `must_not` 로 명확히 표현된다. 문서 메타데이터는 payload의 `metadata.<key>` 경로로 접근.

In [ ]:
from qdrant_client import models

# lang='ko' 이고 source가 'qdrant' 또는 'docs' 인 문서만
flt = models.Filter(
    must=[
        models.FieldCondition(key="metadata.lang", match=models.MatchValue(value="ko")),
    ],
    should=[
        models.FieldCondition(key="metadata.source", match=models.MatchValue(value="qdrant")),
        models.FieldCondition(key="metadata.source", match=models.MatchValue(value="docs")),
    ],
)

hits = store.similarity_search("벡터 엔진 아키텍처", k=5, filter=flt)
for d in hits:
    print("-", d.metadata, "|", d.page_content[:40])

## 3.05.4 점수 포함 · MMR

`similarity_search_with_score` 는 코사인 **유사도**(1.0=최대)를 반환.

In [ ]:
print("--- 유사도 ---")
for doc, sim in store.similarity_search_with_score("Rust 엔진", k=3):
    print(f"{sim:.4f}  {doc.metadata['source']}")

print("\n--- MMR ---")
for d in store.max_marginal_relevance_search("벡터 검색 아키텍처", k=3, fetch_k=4, lambda_mult=0.3):
    print("-", d.metadata["source"])

## 3.05.5 하이브리드 검색 — dense + sparse

Qdrant는 같은 컬렉션 안에 **dense vector**(semantic)와 **sparse vector**(BM25 스타일 keyword)를 함께 저장하고, 검색 시점에 두 결과를 RRF(Reciprocal Rank Fusion)로 섞을 수 있다.

`FastEmbedSparse("Qdrant/bm25")` — BM25 통계 기반 sparse embedding. 별도 학습 불필요.

아래 셀은 `fastembed`(onnxruntime 기반)가 현재 파이썬 인터프리터에서 정상 로드돼야 실행된다. Python 3.14 + 최신 onnxruntime 조합에서는 세그폴트로 커널이 죽는 경우가 있어 가드를 둔다 — 그런 환경에서는 로그만 남기고 넘어가고, 별도로 3.12 환경에서 snippet을 실습한다.

In [ ]:
import sys
# Python 3.14 + onnxruntime 조합에서 FastEmbed가 세그폴트 나는 경우가 있어 가드한다.
HYBRID_OK = sys.version_info < (3, 14)

if HYBRID_OK:
    from langchain_qdrant import FastEmbedSparse, RetrievalMode

    sparse = FastEmbedSparse(model_name="Qdrant/bm25")
    hybrid_store = QdrantVectorStore.from_documents(
        docs,
        embedding=embeddings,
        sparse_embedding=sparse,
        retrieval_mode=RetrievalMode.HYBRID,
        location=":memory:",
        collection_name="vs_hybrid_demo",
    )
    # HNSW(dense) 단독이면 놓칠 수 있는 정확 키워드 질의
    for d in hybrid_store.similarity_search("HNSW", k=3):
        print("-", d.metadata["source"], "|", d.page_content[:50])
else:
    print("fastembed skipped on Python", sys.version_info[:2], "— snippet 참고용")

## 3.05.6 영속성 · 원격 연결

- 로컬 파일 기반 영속화: `QdrantVectorStore.from_documents(..., path='./qdrant_local', collection_name=...)`
- 원격 Qdrant 서버: `QdrantVectorStore(client=QdrantClient(url='https://xxx.qdrant.tech', api_key=...), collection_name=..., embedding=...)`

아래는 같은 프로세스 내에서 파일 경로로 재현하는 예 (노트북 데모용).

In [ ]:
import tempfile, shutil

local_path = tempfile.mkdtemp(prefix="qdrant_")

persisted = QdrantVectorStore.from_documents(
    docs,
    embedding=embeddings,
    path=local_path,
    collection_name="vs_persist",
)
print("persist path:", local_path)
print("count:", persisted.client.count(collection_name="vs_persist").count)

# 로컬 파일 모드는 단일 프로세스 내에서도 락을 잡는다 — 재연결 전에 기존 클라이언트를 닫아야 한다.
persisted.client.close()
del persisted

reopened_client = QdrantClient(path=local_path)
reopened = QdrantVectorStore(
    client=reopened_client,
    collection_name="vs_persist",
    embedding=embeddings,
)
for d in reopened.similarity_search("벡터 엔진", k=2):
    print("-", d.metadata["source"])

reopened_client.close()
shutil.rmtree(local_path)

## 체크리스트

- [ ] `:memory:` 는 프로세스 재시작 시 소실 — 영속화는 `path=` 또는 원격 URL
- [ ] 하이브리드 검색에는 **sparse_embedding + retrieval_mode=HYBRID** 가 모두 필요
- [ ] 필터는 payload 경로 기준 — `metadata.<key>` 로 접근 (langchain 기본 저장 키)
- [ ] 대규모 쓰기에는 gRPC(`prefer_grpc=True`) 가 HTTP보다 빠름

## 다음

- `06_weaviate.ipynb` — 또 다른 하이브리드 엔진
- `05_retrievers/` — RRF · rerank 레이어

## 참고

- Qdrant docs: https://qdrant.tech/documentation/
- `langchain-qdrant`: https://python.langchain.com/docs/integrations/vectorstores/qdrant